
-   Imports



In [0]:
from pyspark.sql import functions as F

- Logging helpers

In [0]:
def log(msg):
    print(f"[INFO] {msg}")

def error(msg):
    print(f"[ERROR] {msg}")

- Reading Bronze tables

In [0]:
try:
    log("Reading Bronze tables")

    source_df = spark.table("dq_project.bronze.source_raw")
    target_df = spark.table("dq_project.bronze.target_raw")

    display(source_df.limit(5))
    display(target_df.limit(5))

    log("Successfully read Bronze tables")

except Exception as e:
    error(f"Failed to read Bronze tables: {str(e)}")
    raise

[INFO] Reading Bronze tables


customer_id,first_name,last_name,gender,city,signup_date,age,annual_income,credit_score,tenure_months,is_active,purchase_count,loyalty_score,risk_band,_rescued_data,dataset_type,ingestion_ts
100001,Diya,Kulkarni,F,Chennai,2020-08-17,40,975535.0,687.0,39,1,15,48.4,Medium,null,source,2026-04-01T19:56:30.35933Z
100002,Vivaan,Mehta,M,Mumbai,2021-12-03,35,1063001.0,685.0,21,0,13,77.7,Medium,null,source,2026-04-01T19:56:30.35933Z
100003,Anaya,Nair,F,Bengaluru,2023-11-28,42,777112.0,611.0,40,1,12,63.2,High,null,source,2026-04-01T19:56:30.35933Z
100004,Aisha,Malhotra,F,Kolkata,2020-05-23,50,715308.0,692.0,82,1,10,89.6,Medium,null,source,2026-04-01T19:56:30.35933Z
100005,Anaya,Mishra,M,null,2020-03-08,34,611267.0,750.0,54,1,16,78.7,Low,null,source,2026-04-01T19:56:30.35933Z


customer_id,first_name,last_name,gender,city,signup_date,age,annual_income,credit_score,tenure_months,is_active,purchase_count,loyalty_score,preferred_channel,_rescued_data,dataset_type,ingestion_ts
100001,Diya,Kulkarni,F,Chennai,2020-08-17,40,1080183.0,689.0,39,1,17,48.3,App,null,target,2026-04-01T19:57:17.538275Z
100002,Vivaan,Mehta,M,Mumbai,2021-12-03,35,1110299.0,674.0,21,0,16,81.0,Email,null,target,2026-04-01T19:57:17.538275Z
100003,Anaya,Nair,F,null,2023-11-28,42,841204.0,600.0,40,1,14,59.7,App,null,target,2026-04-01T19:57:17.538275Z
100004,Aisha,Malhotra,F,Kolkata,2020-05-23,50,754164.0,680.0,82,1,12,93.6,App,null,target,2026-04-01T19:57:17.538275Z
100005,Anaya,Mishra,M,null,2020-03-08,34,663744.0,734.0,54,1,17,78.9,Email,null,target,2026-04-01T19:57:17.538275Z


[INFO] Successfully read Bronze tables


- Doing Row Count Comparisons

In [0]:
try:
    log("Calculating row counts")

    source_count = source_df.count()
    target_count = target_df.count()

    row_count_df = spark.createDataFrame(
        [(source_count, target_count, target_count - source_count)],
        ["source_count", "target_count", "difference"]
    )

    display(row_count_df)

    log("Row count comparison completed")

except Exception as e:
    error(f"Row count comparison failed: {str(e)}")
    raise

[INFO] Calculating row counts


source_count,target_count,difference
2500,2520,20


[INFO] Row count comparison completed


- Saving Row Counts

In [0]:
try:
    log("Saving row_count_diff table")

    row_count_df.write.mode("overwrite").saveAsTable("dq_project.silver.row_count_diff")

    log("row_count_diff saved successfully")

except Exception as e:
    error(f"Saving row_count_diff failed: {str(e)}")
    raise

[INFO] Saving row_count_diff table
[INFO] row_count_diff saved successfully


- Doing Column Comparisons

In [0]:
try:
    log("Comparing schemas")

    source_cols = set(source_df.columns)
    target_cols = set(target_df.columns)

    only_in_source = list(source_cols - target_cols)
    only_in_target = list(target_cols - source_cols)

    schema_diff_data = []

    for c in only_in_source:
        schema_diff_data.append((c, "only_in_source"))

    for c in only_in_target:
        schema_diff_data.append((c, "only_in_target"))

    schema_diff_df = spark.createDataFrame(schema_diff_data, ["column_name", "difference"])

    display(schema_diff_df)

    log("Schema comparison completed")

except Exception as e:
    error(f"Schema comparison failed: {str(e)}")
    raise

[INFO] Comparing schemas


column_name,difference
risk_band,only_in_source
preferred_channel,only_in_target


[INFO] Schema comparison completed


- Saving Schema Diffrences

In [0]:
try:
    log("Saving schema_diff table")

    schema_diff_df.write.mode("overwrite").saveAsTable("dq_project.silver.schema_diff")

    log("schema_diff saved successfully")

except Exception as e:
    error(f"Saving schema_diff failed: {str(e)}")
    raise

[INFO] Saving schema_diff table
[INFO] schema_diff saved successfully


- Doing Null Comparisons

In [0]:
try:
    log("Checking null values")

    common_cols = list(source_cols.intersection(target_cols))

    null_data = []

    for c in common_cols:
        src_null = source_df.filter(F.col(c).isNull()).count()
        tgt_null = target_df.filter(F.col(c).isNull()).count()

        null_data.append((c, src_null, tgt_null))

    null_df = spark.createDataFrame(null_data, ["column", "source_nulls", "target_nulls"])

    display(null_df)

    log("Null comparison completed")

except Exception as e:
    error(f"Null comparison failed: {str(e)}")
    raise

[INFO] Checking null values


column,source_nulls,target_nulls
loyalty_score,0,0
dataset_type,0,0
ingestion_ts,0,0
first_name,0,0
tenure_months,0,0
city,65,158
last_name,0,0
gender,0,0
is_active,0,0
customer_id,0,0


[INFO] Null comparison completed


- Saving Null Diffrences

In [0]:
try:
    log("Saving null_diff table")

    null_df.write.mode("overwrite").saveAsTable("dq_project.silver.null_diff")

    log("null_diff saved successfully")

except Exception as e:
    error(f"Saving null_diff failed: {str(e)}")
    raise

[INFO] Saving null_diff table
[INFO] null_diff saved successfully


- Identifying numeric columns

In [0]:
numeric_cols = [c for c, t in source_df.dtypes if t in ["int", "double", "bigint"]]

print("Numeric Columns:", numeric_cols)

Numeric Columns: ['customer_id', 'age', 'annual_income', 'credit_score', 'tenure_months', 'is_active', 'purchase_count', 'loyalty_score']


- Doing Average Comparisons

In [0]:
try:
    log("Calculating average comparison")

    avg_data = []

    for c in numeric_cols:
        src_avg = source_df.select(F.avg(c)).collect()[0][0]
        tgt_avg = target_df.select(F.avg(c)).collect()[0][0]

        src_avg_val = float(src_avg) if src_avg is not None else None
        tgt_avg_val = float(tgt_avg) if tgt_avg is not None else None

        diff = None
        if src_avg_val is not None and tgt_avg_val is not None:
            diff = tgt_avg_val - src_avg_val

        avg_data.append((c, src_avg_val, tgt_avg_val, diff))

    avg_df = spark.createDataFrame(
        avg_data,
        ["column", "source_avg", "target_avg", "avg_difference"]
    )

    display(avg_df)

except Exception as e:
    error(f"Average comparison failed: {str(e)}")
    raise

[INFO] Calculating average comparison


column,source_avg,target_avg,avg_difference
customer_id,101250.5,101260.5,10.0
age,36.4576,36.46626984126984,0.008669841269842493
annual_income,845893.8056910569,904966.2390143737,59072.43332331686
credit_score,708.3317191283293,696.5274193548387,-11.804299773490584
tenure_months,48.8756,48.801984126984124,-0.07361587301587491
is_active,0.788,0.7880952380952381,9.523809523803717E-5
purchase_count,13.9792,15.638095238095238,1.658895238095237
loyalty_score,73.86568,75.33083333333336,1.4651533333333617


- Saving Average Comparisons

In [0]:
avg_df.write.mode("overwrite").saveAsTable("dq_project.silver.avg_diff")

- Doing Min Comparisons

In [0]:
try:
    log("Calculating min comparison")

    min_data = []

    for c in numeric_cols:
        src_min = source_df.select(F.min(c)).collect()[0][0]
        tgt_min = target_df.select(F.min(c)).collect()[0][0]

        src_min_val = float(src_min) if src_min is not None else None
        tgt_min_val = float(tgt_min) if tgt_min is not None else None

        diff = None
        if src_min_val is not None and tgt_min_val is not None:
            diff = tgt_min_val - src_min_val

        min_data.append((c, src_min_val, tgt_min_val, diff))

    min_df = spark.createDataFrame(
        min_data,
        ["column", "source_min", "target_min", "min_difference"]
    )

    display(min_df)

except Exception as e:
    error(f"Min comparison failed: {str(e)}")
    raise

[INFO] Calculating min comparison


column,source_min,target_min,min_difference
customer_id,100001.0,100001.0,0.0
age,21.0,21.0,0.0
annual_income,250000.0,251271.0,1271.0
credit_score,540.0,530.0,-10.0
tenure_months,1.0,1.0,0.0
is_active,0.0,0.0,0.0
purchase_count,3.0,3.0,0.0
loyalty_score,30.6,32.0,1.3999999999999986


- Saving Mean Comparisons

In [0]:
min_df.write.mode("overwrite").saveAsTable("dq_project.silver.min_diff")

- Doing Max Comparisons

In [0]:
try:
    log("Calculating max comparison")

    max_data = []

    for c in numeric_cols:
        src_max = source_df.select(F.max(c)).collect()[0][0]
        tgt_max = target_df.select(F.max(c)).collect()[0][0]

        src_max_val = float(src_max) if src_max is not None else None
        tgt_max_val = float(tgt_max) if tgt_max is not None else None

        diff = None
        if src_max_val is not None and tgt_max_val is not None:
            diff = tgt_max_val - src_max_val

        max_data.append((c, src_max_val, tgt_max_val, diff))

    max_df = spark.createDataFrame(
        max_data,
        ["column", "source_max", "target_max", "max_difference"]
    )

    display(max_df)

except Exception as e:
    error(f"Max comparison failed: {str(e)}")
    raise

[INFO] Calculating max comparison


column,source_max,target_max,max_difference
customer_id,102500.0,102520.0,20.0
age,70.0,70.0,0.0
annual_income,1713772.0,1893107.0,179335.0
credit_score,850.0,848.0,-2.0
tenure_months,96.0,96.0,0.0
is_active,1.0,1.0,0.0
purchase_count,27.0,32.0,5.0
loyalty_score,100.0,100.0,0.0


- Saving Max Comparisons

In [0]:
max_df.write.mode("overwrite").saveAsTable("dq_project.silver.max_diff")

- Doing Count Comparisons

In [0]:
try:
    log("Calculating count comparison")

    count_data = []

    for c in numeric_cols:
        src_count = source_df.select(F.count(c)).collect()[0][0]
        tgt_count = target_df.select(F.count(c)).collect()[0][0]

        diff = tgt_count - src_count

        count_data.append((c, src_count, tgt_count, diff))

    count_df = spark.createDataFrame(
        count_data,
        ["column", "source_count", "target_count", "count_difference"]
    )

    display(count_df)

except Exception as e:
    error(f"Count comparison failed: {str(e)}")
    raise

[INFO] Calculating count comparison


column,source_count,target_count,count_difference
customer_id,2500,2520,20
age,2500,2520,20
annual_income,2460,2435,-25
credit_score,2478,2480,2
tenure_months,2500,2520,20
is_active,2500,2520,20
purchase_count,2500,2520,20
loyalty_score,2500,2520,20


Saving Count Comparisons

In [0]:
count_df.write.mode("overwrite").saveAsTable("dq_project.silver.count_diff")

- Key Checking

In [0]:
try:
    log("Checking primary key duplicates")

    key = "customer_id"

    src_dup = source_df.groupBy(key).count().filter("count > 1").count()
    tgt_dup = target_df.groupBy(key).count().filter("count > 1").count()

    key_df = spark.createDataFrame(
        [(key, src_dup, tgt_dup)],
        ["key_column", "source_duplicates", "target_duplicates"]
    )

    display(key_df)

    log("Key check completed")

except Exception as e:
    error(f"Key check failed: {str(e)}")
    raise

[INFO] Checking primary key duplicates


key_column,source_duplicates,target_duplicates
customer_id,0,0


[INFO] Key check completed


- Saving Key Checking table

In [0]:
try:
    log("Saving key_quality table")

    key_df.write.mode("overwrite").saveAsTable("dq_project.silver.key_quality")

    log("key_quality saved successfully")

except Exception as e:
    error(f"Saving key_quality failed: {str(e)}")
    raise

[INFO] Saving key_quality table
[INFO] key_quality saved successfully
